# Fine-Tuning BERT for Sentiment Classification

Notebook này trình bày cách fine-tune mô hình BERT cho bài toán phân loại cảm xúc văn bản.

Mục tiêu là xây dựng một pipeline hoàn chỉnh từ dữ liệu đến huấn luyện và dự đoán.

---

## Giới thiệu

BERT là một mô hình ngôn ngữ được huấn luyện trước bằng phương pháp Masked Language Model, cho phép học được biểu diễn ngữ nghĩa và ngữ cảnh hai chiều của từ.

Thay vì huấn luyện từ đầu, ta có thể tận dụng kiến thức đã học bằng cách fine-tune mô hình cho một bài toán cụ thể.

Trong bài toán này, đầu vào là một câu văn bản và đầu ra là nhãn cảm xúc (positive hoặc negative).

---

## Cài đặt và môi trường

Sử dụng các thư viện:

- transformers: cung cấp mô hình và Trainer API
- datasets: hỗ trợ load và xử lý dữ liệu
- torch: framework deep learning

Cần đảm bảo các thư viện được cài đặt đúng phiên bản và kernel đã được khởi động lại.

---

## Dataset

Dataset IMDB gồm 50,000 review phim, chia thành tập train và test.

Mỗi mẫu gồm:
- Văn bản (review)
- Nhãn (0: negative, 1: positive)

Đây là bài toán phân loại nhị phân.

---

## Tokenization

Văn bản không thể đưa trực tiếp vào mô hình, cần được chuyển thành token IDs.

Tokenizer thực hiện:
- Tách từ thành subword
- Thêm token đặc biệt như [CLS] và [SEP]
- Padding và truncation

Token [CLS] đóng vai trò đại diện cho toàn bộ câu và sẽ được dùng cho classification.

---

## Mô hình

Sử dụng AutoModelForSequenceClassification dựa trên BERT.

Kiến trúc:
Input text → BERT encoder → vector [CLS] → lớp tuyến tính → xác suất nhãn

Số nhãn là 2 tương ứng với positive và negative.

---

## Huấn luyện

Sử dụng Trainer API để huấn luyện mô hình.

Quá trình huấn luyện gồm:
- Forward pass
- Tính loss (cross-entropy)
- Backpropagation
- Cập nhật trọng số

Để giảm thời gian, chỉ sử dụng một phần nhỏ dataset.

Evaluation được thực hiện sau mỗi epoch.

---

## Đánh giá

Kết quả evaluation được thực hiện tự động trong quá trình training.

Các chỉ số chính:
- Train loss
- Eval loss

Eval loss thấp cho thấy mô hình có khả năng tổng quát hóa tốt.

Kết quả có thể truy cập thông qua trainer.state.log_history.

---

## Dự đoán

Sau khi huấn luyện, mô hình có thể được sử dụng để dự đoán dữ liệu mới.

Pipeline hỗ trợ:
- Tokenization tự động
- Dự đoán nhanh
- Trả về nhãn và xác suất

---

## Phân tích

Fine-tuning cho phép mô hình thích nghi nhanh với bài toán mới nhờ kiến thức từ pretraining.

Mô hình hoạt động tốt ngay cả với dữ liệu nhỏ và số epoch thấp.

BERT có khả năng hiểu ngữ cảnh tốt hơn các phương pháp truyền thống.

---

## Kết luận

Notebook đã trình bày quy trình fine-tuning BERT cho bài toán phân loại văn bản.

Các điểm chính:
- Sử dụng pretrained model giúp tiết kiệm tài nguyên
- Fine-tuning mang lại hiệu quả cao
- Pipeline đơn giản nhưng mạnh mẽ

---

## Hướng phát triển

- Áp dụng cho bài toán multi-class
- Fine-tune trên dữ liệu tiếng Việt
- Thêm các metric như accuracy, F1-score
- Phân tích lỗi chi tiết hơn

In [1]:
!pip install "transformers[torch]" datasets

In [2]:
from datasets import load_dataset

dataset = load_dataset("imdb")

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize, batched=True)

In [4]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8223.15it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider tr

In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=dataset["test"].select(range(500))
)

trainer.train()

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]


TrainOutput(global_step=250, training_loss=0.5111802062988281, metrics={'train_runtime': 734.6422, 'train_samples_per_second': 2.722, 'train_steps_per_second': 0.34, 'total_flos': 526222110720000.0, 'train_loss': 0.5111802062988281, 'epoch': 1.0})

In [8]:
print(trainer.state)

TrainerState(epoch=1.0, global_step=250, max_steps=250, logging_steps=500, eval_steps=500, save_steps=500, train_batch_size=8, num_train_epochs=1, num_input_tokens_seen=0, total_flos=526222110720000.0, log_history=[{'train_runtime': 734.6422, 'train_samples_per_second': 2.722, 'train_steps_per_second': 0.34, 'total_flos': 526222110720000.0, 'train_loss': 0.5111802062988281, 'epoch': 1.0, 'step': 250}, {'eval_loss': 0.23624302446842194, 'eval_runtime': 45.6632, 'eval_samples_per_second': 10.95, 'eval_steps_per_second': 1.38, 'epoch': 1.0, 'step': 250}], best_metric=None, best_global_step=None, best_model_checkpoint=None, is_local_process_zero=True, is_world_process_zero=True, is_hyper_param_search=False, trial_name=None, trial_params=None, stateful_callbacks={'TrainerControl': {'args': {'should_training_stop': True, 'should_epoch_stop': False, 'should_save': True, 'should_evaluate': False, 'should_log': False}, 'attributes': {}}})


In [9]:
trainer.state.log_history

[{'train_runtime': 734.6422,
  'train_samples_per_second': 2.722,
  'train_steps_per_second': 0.34,
  'total_flos': 526222110720000.0,
  'train_loss': 0.5111802062988281,
  'epoch': 1.0,
  'step': 250},
 {'eval_loss': 0.23624302446842194,
  'eval_runtime': 45.6632,
  'eval_samples_per_second': 10.95,
  'eval_steps_per_second': 1.38,
  'epoch': 1.0,
  'step': 250}]

In [11]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

print(classifier("This product is amazing!"))
print(classifier("I wasted my money."))

[{'label': 'LABEL_1', 'score': 0.9795374870300293}]
[{'label': 'LABEL_0', 'score': 0.9483171701431274}]
